# General AC-MOT G1 — runtime on a Colab T4

Times the frozen G1 candidate (tag `general-acmot-g1-freeze`) on real VisDrone2019-MOT-val frames: host alone, G1 (V7f at 736 px), and the historical SCI arm, for YOLOv8n, RT-DETR-L and RetinaNet; plus the detector latency curve from 512 to 960 px. Stages are CUDA-synchronised. Runtime only: no metric is computed.

Runtime → Change runtime type → T4 GPU before running.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# find VisDrone2019-MOT-val (folder or zip) on Drive
find /content/drive -maxdepth 6 \( -name 'VisDrone2019-MOT-val.zip' -o -type d -name 'VisDrone2019-MOT-val' \) 2>/dev/null | head

In [ ]:
VAL = '/content/drive/MyDrive/PATH/TO/VisDrone2019-MOT-val'   # folder with sequences/ (or set VAL_ZIP below)
VAL_ZIP = ''   # e.g. '/content/drive/MyDrive/.../VisDrone2019-MOT-val.zip'

In [ ]:
%%bash -s "$VAL_ZIP"
if [ -n "$1" ]; then
  echo "e53571990dfc79229e0a8ae10264bc4fa604a027c44b06e3a097417e4fa55705  $1" | sha256sum -c -
  mkdir -p /content/val && unzip -q "$1" '*uav0000086_00000_v*' '*uav0000339_00001_v*' -d /content/val
  find /content/val -type d -name sequences
fi

In [ ]:
%%bash
set -e
git clone -q -b sci-v7f-general-layer-dev https://github.com/AhmedCode110/AC-MOT.git /content/AC-MOT
cd /content/AC-MOT && git fetch -q --tags && git checkout -q general-acmot-g1-freeze && git log --oneline -1
pip -q install ultralytics==8.3.200 lap motmetrics==1.4.0 scipy opencv-python-headless
cd /content
curl -fsSL -o yolov8n.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt
curl -fsSL -o rtdetr-l.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/rtdetr-l.pt
curl -fsSL -o retinanet_resnet50_fpn_v2_coco-5905b1c5.pth https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
sha256sum yolov8n.pt rtdetr-l.pt retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
python -c 'import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))'

Expected sha256: yolov8n `f59b3d83…`, rtdetr-l `6de60b10…`, RetinaNet `5905b1c5…`. The benchmark scripts come from the tag `general-acmot-g1-freeze` checkout; `tools/sci_v7/benchmark.py` and `build_sweep_cache.py` are read from the branch head if the tag predates `--max-frames` (next cell).

In [ ]:
%%bash
cd /content/AC-MOT && git show sci-v7f-general-layer-dev:tools/sci_v7/benchmark.py > /tmp/benchmark.py 2>/dev/null || git show origin/sci-v7f-general-layer-dev:tools/sci_v7/benchmark.py > /tmp/benchmark.py
git show origin/sci-v7f-general-layer-dev:tools/sci_v7/build_sweep_cache.py > tools/sci_v7/build_sweep_cache.py
cp /tmp/benchmark.py tools/sci_v7/benchmark.py && git status --short

In [ ]:
import glob, os, subprocess
DATA = VAL if not VAL_ZIP else os.path.dirname(glob.glob('/content/val/**/sequences', recursive=True)[0])
OUT = '/content/drive/MyDrive/G1_T4_runtime'; os.makedirs(OUT, exist_ok=True)
W = {'yolov8': '/content/yolov8n.pt', 'rtdetr': '/content/rtdetr-l.pt', 'retinanet': '/content/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth'}
for det, w in W.items():
    for name, arms, seqs, n in [('bench', 'native:736 fixed:736 sci:resolution', 'uav0000086_00000_v uav0000339_00001_v', 0),
                                ('latency_curve', ' '.join(f'fixed:{r}' for r in (512,576,640,704,736,768,832,896,960)), 'uav0000086_00000_v', 120)]:
        cmd = (f'cd /content/AC-MOT && PYTHONPATH=. python tools/sci_v7/benchmark.py --weights {w} --det {det} --dataset {DATA} '
               f'--seqs {seqs} --arms {arms} --out {OUT}/{name}_{det}_t4.json' + (f' --max-frames {n}' if n else ''))
        print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-1500:])

In [ ]:
import json, glob
for f in sorted(glob.glob(OUT + '/bench_*_t4.json')):
    r = json.load(open(f)); print(r['det'], r['hardware']['gpu'])
    for a, v in r['arms'].items():
        print(f"  {a:<16} total {v['total']['mean_ms']:.1f} ms (p95 {v['total']['p95_ms']:.1f})  detector {v['detector']['mean_ms']:.1f}  controller {v['controller_ms']['mean']:.2f} ms  FPS {v['fps_excl_decode']:.1f}")

Copy the JSON files from `MyDrive/G1_T4_runtime/` into `research/final/sci_v7f/G1_runtime_t4/` of the repository (or send them back) so they enter the record.